In [1]:
# test with supervised optimal transport loss
# comparison with unsupervised

import random
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.utils.data import DataLoader
import deeplake
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score
from sklearn.decomposition import PCA
from geomloss import SamplesLoss
import time
import pandas as pd
from torch.utils.data import ConcatDataset, RandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
import pickle
import seaborn as sns
import itertools
from dataset_OT import make_multi_WSI_dataset

seed = 42
torch.manual_seed(seed)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False 


batch_size = 512 

print('Loading starting...')
idx_range_subset1 = [i for i in range(1,52+1)]
random.shuffle(idx_range_subset1)
num_train = int(np.ceil(0.7 * len(idx_range_subset1))) 
train_range1, val_range1 = idx_range_subset1[:num_train], idx_range_subset1[num_train:]

idx_range_subset3 = [i for i in range(1,26+1)] 
random.shuffle(idx_range_subset3)
num_train = int(np.ceil(0.7 * len(idx_range_subset3))) 
train_range3, val_range3 = idx_range_subset3[:num_train], idx_range_subset3[num_train:]

akoya_loader_train_subset1 = make_multi_WSI_dataset('Subset1', train_range1, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_val_subset1 = make_multi_WSI_dataset('Subset1', val_range1, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_train_subset3 = make_multi_WSI_dataset('Subset3', train_range3, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_val_subset3 = make_multi_WSI_dataset('Subset3', val_range3, ['Akoya'], train_or_test='Train', batch_size=batch_size)
leica_loader_train = make_multi_WSI_dataset('Subset3', train_range3, ['Leica'], train_or_test='Train', batch_size=batch_size)
leica_loader_val = make_multi_WSI_dataset('Subset3', val_range3, ['Leica'], train_or_test='Train', batch_size=batch_size)

akoya_loader_train = ConcatDataset([akoya_loader_train_subset1, akoya_loader_train_subset3])
akoya_loader_val = ConcatDataset([akoya_loader_val_subset1, akoya_loader_val_subset3])

len_akoya_train = len(akoya_loader_train)
len_akoya_val = len(akoya_loader_val)

len_leica_train = len(leica_loader_train)
len_leica_val = len(leica_loader_val)

len_train = len_akoya_train + len_leica_train
len_val = len_akoya_val + len_leica_val

B_A_train = round(batch_size * len_akoya_train / (len_akoya_train + len_leica_train))
B_L_train = batch_size - B_A_train 
B_A_val = round(batch_size * len_akoya_val / (len_akoya_val + len_leica_val))
B_L_val = batch_size - B_A_val

akoya_loader_train = DataLoader(akoya_loader_train, batch_size=B_A_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
akoya_loader_val = DataLoader(akoya_loader_val, batch_size=B_A_val, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
leica_loader_train = DataLoader(leica_loader_train, batch_size=B_L_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
leica_loader_val = DataLoader(leica_loader_val, batch_size=B_L_val, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)


print("Train batches Akoya:", len(akoya_loader_train), 'batch size:', B_A_train)
print("Train batches Leica:", len(leica_loader_train), 'batch size:', B_L_train)
print("Validation batches Akoya:", len(akoya_loader_val), 'batch size:', B_A_val)
print("Validation batches Leica:", len(leica_loader_val), 'batch size:', B_L_val)

#in samples
print('len train:', len_train)



Loading starting...
Train batches Akoya: 4973 batch size: 399
Train batches Leica: 4978 batch size: 113
Validation batches Akoya: 1272 batch size: 388
Validation batches Leica: 1267 batch size: 124
len train: 2546444


In [2]:
# unsupervised
unsupervised_OT = SamplesLoss('sinkhorn', p=2, blur=0.1, scaling=0.95, verbose=False)



## Selective optimal transport

## test

In [3]:
def make_cost_fn(labels_x, labels_y, p=100.0):
    """
    Create a custom cost function with label penalties for SamplesLoss.
    """

    def cost_fn(x, y):
        N = x.shape[1]
        M = y.shape[1]

        x_norm = (x ** 2).sum(dim=2, keepdim=True)        # (B,N,1)
        y_norm = (y ** 2).sum(dim=2, keepdim=True)        # (B,M,1)
        xy = torch.bmm(x, y.transpose(1, 2))              # (B,N,M)
        dist = 0.5 * (x_norm + y_norm.transpose(1, 2) - 2 * xy)
        dist = torch.clamp_(dist, min=0.0)

        #SamplesLoss compute the cost for x,y / y,x / x,x / y,y --> careful with shapes
        if N == len(labels_x) and M == len(labels_y):
            lx, ly = labels_x, labels_y
            
        elif N == len(labels_y) and M == len(labels_x):
            lx, ly = labels_y, labels_x
            
        else:
            #when same vectors x or y are used, necessarily the labels will align, no need for penalty
            return dist

        lx = lx.to(x.device, non_blocking=True)
        ly = ly.to(y.device, non_blocking=True)

        lx = lx.unsqueeze(0).unsqueeze(2)        # (1,N,1)
        ly = ly.unsqueeze(0).unsqueeze(0)        # (1,1,M)
        S = (lx != ly).float().expand(1, -1, -1) # (B=1,N,M)
        #-1 means 'keep the size of this dimension dont copy'

        return dist + p * S

    return cost_fn

def check_label_transport(cost_fn, x, y, labels_x, labels_y, penalty=100.0, blur=0.05):
    """
    Check if the cost function encourages transport between points with same labels.
    Works with the batched (B,N,D) format expected by make_cost_fn.
    """
    if x.dim() == 2:   # (N,D) → add batch dim
        x = x.unsqueeze(0)   # (1,N,D)
    if y.dim() == 2:   # (M,D)
        y = y.unsqueeze(0)   # (1,M,D)

    B, N, D = x.shape
    _, M, _ = y.shape

    # Uniform weights with batch
    a = torch.ones(B, N, device=x.device) / N   # (B,N)
    b = torch.ones(B, M, device=y.device) / M   # (B,M)

    # Sinkhorn with dual potentials
    loss = SamplesLoss(loss="sinkhorn", p=2, blur=0.05, 
                           backend="tensorized", cost=cost_fn, potentials=True)
    F, G = loss(a, x, b, y)  # F: (B,N), G: (B,M)

    # Remove batch dimension
    F = F[0]  # (N,)
    G = G[0]  # (M,)

    # Cost matrix (B,N,M) → (N,M)
    C = cost_fn(x, y)[0]

    # Regularization parameter (same as blur)
    ε = blur  

    # Gibbs kernel
    K = torch.exp(-(C - F.unsqueeze(1) - G.unsqueeze(0)) / ε)  # (N,M)

    # Transport plan
    P = (a[0].unsqueeze(1) * K * b[0].unsqueeze(0))
    P = P / P.sum()  # normalize to sum=1

    # Same-label mask
    lx = labels_x.unsqueeze(1).expand(N, M)
    ly = labels_y.unsqueeze(0).expand(N, M)
    mask = (lx == ly).float()

    # Fraction of transport mass that goes to same-label pairs
    fraction_same_label = (P * mask).sum().item()

    #print("Total transport mass (sanity check):", P.sum().item())
    #print("Transport mass to same-label pairs:", fraction_same_label)

    return fraction_same_label




    

In [4]:
go = False
lu_OT = []
ls_OT = []
sup_times, unsup_times = [], []
quality = []
avg_sup = 0
avg_unsup = 0
device = torch.device('cuda')

p_penalty = 1000



if go:
    tq = tqdm(zip(akoya_loader_train, leica_loader_train),
                                                    desc=f"Timing OT losses",
                                                    total=min(len(akoya_loader_train), len(leica_loader_train)))
    total_time = 0
    for batch_akoya, batch_leica in tq:
                
        embedding_akoya = batch_akoya['embedding'].to(device, non_blocking=True)
        embedding_leica = batch_leica['embedding'].to(device, non_blocking=True)
        labels_akoya = batch_akoya['label'].to(device, non_blocking=True)
        labels_leica = batch_leica['label'].to(device, non_blocking=True)
        
        
        # supervised OT
        t0 = time.time()
        cost_fn_high = make_cost_fn(labels_akoya, labels_leica, p=p_penalty)
        loss_high = SamplesLoss(loss="sinkhorn", p=2, blur=0.05, 
                           backend="tensorized", cost=cost_fn_high)
        s_OT = loss_high(embedding_akoya, embedding_leica).cpu().detach().item()
        ls_OT.append(s_OT)
        frac = check_label_transport(cost_fn_high, embedding_akoya, embedding_leica, 
                             labels_akoya, labels_leica)
        quality.append(frac)
        t1 = time.time()
        sup_times.append(t1 - t0)
        
        # unsupervised OT
        '''t2 = time.time()
        u_OT = unsupervised_OT(embedding_akoya, embedding_leica).cpu().detach().item()
        lu_OT.append(u_OT)
        t3 = time.time()
        unsup_times.append(t3 - t2)'''
        
        # update tqdm description with average time and quality
        avg_sup = sum(sup_times) / len(sup_times)
        avg_quality = sum(quality) / len(quality)
        tq.set_description(f"Timing OT losses | Avg sup: {avg_sup:.3f}s | Avg quality: {avg_quality:.3f}s")
                                                                                 
    print('finish')

## Quality check

In [5]:
import torch
import torch.nn as nn
import random

class Network(nn.Module):
    """
    Initialises an Artificial Neural Network with the foundation encoder Gigapath 
    and 2 layers for classification (one to create embeddings, one to classify)

    """

    def __init__(self, emb_mode: bool = False, freeze_encoder: bool = True, OT: bool = False, num_classes: int = 5):
        super().__init__() #super constructor for ANN in PyTorch

        self.freeze_encoder = freeze_encoder
        self.OT = OT #maybe not useful here
        self.emb_mode = emb_mode
        
        # Define encoder

        if not self.emb_mode:
            encoder_name = 'gigapath'
            BASE_MODEL_DIR = '/home/leolr-int/AGGCPerturbations/model_weights'
            encoder_dir = os.path.join(BASE_MODEL_DIR, "pre_trained_weights")
            encoder_path = os.path.join(encoder_dir, f"{encoder_name}.pth")
            encoder = torch.load(encoder_path, map_location=torch.device("cpu"), weights_only=False)
            self.encoder = encoder

            if self.freeze_encoder:
                for param in self.encoder.parameters():
                    param.requires_grad = False
        
        else:
            #no need for the encoder
            self.encoder = None #I didnt know we could do this

        # Define bottle neck / embeddings
        # fixed parameter value for Gigapath
        in_dim = 1536
        self.bottle_neck = nn.Sequential(
            nn.Linear(in_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(), #do not forget ReLU
            nn.Dropout(p=0.5))

        # Define classification head
        out_dim = num_classes
        self.head = nn.Linear(1024, out_dim)

    # Define sequential architecture
    def forward(self, x): 
        if self.emb_mode:
            #x is  a vector here
            embedding = self.bottle_neck(x)
        else:
            #x is an image here
            if self.freeze_encoder: 
                with torch.no_grad():
                    encoded = self.encoder(x)
            else:
                encoded = self.encoder(x)
            embedding = self.bottle_neck(encoded)
        logits = self.head(embedding)
        return logits

class NetworkHandler:
    '''
    A class to handle training, inference and prediction
    '''

    def __init__(self, precision = 'mixed', freeze_encoder = True, emb_mode = False, display = False):
        self.precision = precision
        self.freeze_encoder = freeze_encoder
        self.emb_mode = emb_mode
        self.display = display

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = Network(emb_mode=self.emb_mode)
        self.model = self.model.to(self.device, non_blocking=True)

        #printing architecture
        '''print("Bottleneck layers:")
        print(self.model.bottle_neck)
        print("Head layer:")
        print(self.model.head)
        if self.model.encoder is not None:
            print("\nEncoder architecture:")
            print(self.model.encoder)'''

        self.use_amp = precision == 'mixed' and self.device == 'cuda'
        self.grad_scaler = GradScaler(enabled=self.use_amp)
        
    @torch.no_grad()
    def quality_tracking(self, custom_name, p_penalty):
        weights = f'/home/leolr-int/nfs/transformed_data/weights/{custom_name}/checkpoint.pth'
        checkpoint = torch.load(weights, weights_only=False, map_location=self.device)
        #print(checkpoint.keys())
        handler.model.load_state_dict(checkpoint["model"])

        tq = tqdm(zip(akoya_loader_train, leica_loader_train),
                                                    desc=f"Timing OT losses",
                                                    total=min(len(akoya_loader_train), len(leica_loader_train)))

    
        total_time = 0
        for batch_akoya, batch_leica in tq:
                    
            embedding_akoya = batch_akoya['embedding'].to(device, non_blocking=True)
            embedding_leica = batch_leica['embedding'].to(device, non_blocking=True)
            labels_akoya = batch_akoya['label'].to(device, non_blocking=True)
            labels_leica = batch_leica['label'].to(device, non_blocking=True)

            
            with torch.autocast(device_type=self.device, dtype=torch.float16, enabled=self.use_amp):
                embedding_akoya = self.model.bottle_neck(embedding_akoya)
                embedding_leica = self.model.bottle_neck(embedding_leica)
                #dim 1024
            
            
            # supervised OT
            t0 = time.time()
            cost_fn_high = make_cost_fn(labels_akoya, labels_leica, p=p_penalty)
            loss_high = SamplesLoss(loss="sinkhorn", p=2, blur=0.05, 
                               backend="tensorized", cost=cost_fn_high)
            s_OT = loss_high(embedding_akoya.float(), embedding_leica.float()).cpu().detach().item()
            ls_OT.append(s_OT)
            #added float to avoid overflow
            frac = check_label_transport(cost_fn_high, embedding_akoya, embedding_leica, 
                                 labels_akoya, labels_leica)
            quality.append(frac)
            t1 = time.time()
            sup_times.append(t1 - t0)
            
            
            # update tqdm description with average time and quality
            avg_sup = sum(sup_times) / len(sup_times)
            avg_quality = sum(quality) / len(quality)
            tq.set_description(f"OT losses | Avg time: {avg_sup:.3f}s | Avg quality: {avg_quality:.3f}s")
        
        print('finish')
            

In [6]:
testtest = False:
if testtest:
    p_penalty = 5
        
    handler = NetworkHandler(emb_mode=True)
    
    custom_name = 'baseline'
    
    handler.quality_tracking(custom_name, p_penalty)

OT losses | Avg time: 0.024s | Avg quality: 0.728s: 100%|██████████████████████████| 4973/4973 [02:10<00:00, 38.04it/s]

finish


## test with MMOOT


In [ ]:
def pairwise_sqdist(x, y):
    # x: (n,d), y: (m,d)
    x2 = (x**2).sum(dim=1).unsqueeze(1)  # (n,1)
    y2 = (y**2).sum(dim=1).unsqueeze(0)  # (1,m)
    xy = x @ y.T                      # (n,m)
    return x2 + y2 - 2*xy             # (n,m)


def sinkhorn_parallel(A, B, C, eps=1e-2, max_iters=20):
    """
    Parallel Sinkhorn for 4 scanners vs reference Akoya.
    
    Args:
        A: (n, 4) tensor, each column = Akoya histogram (duplicated 4 times)
        B: (n, 4) tensor, columns = histograms of Leica, Philips, Olympus, Zeiss
        C: (n, n) tensor, cost matrix (e.g. squared Euclidean distances)
        eps: float, entropic regularization parameter
        max_iters: int, number of Sinkhorn iterations
        tol: float, stopping criterion on updates
    
    Returns:
        U, V: scaling factors (n,4)
        c: regularized transport costs (4,)
    """
    n, S = A.shape
    K = torch.exp(-C / eps)  # (n, n)

    V = torch.ones_like(B)   # (n, S)

    for _ in range(max_iters):
        U = A / (K @ V + 1e-12)
        V = B / (K.t() @ U + 1e-12)

    # Compute cost
    KC = K * C  # (n, n)
    term1 = U * torch.log(U + 1e-12) * (KC @ V)
    term2 = U * (KC @ (V * torch.log(V + 1e-12)))
    c = term1.sum(dim=0) + term2.sum(dim=0)  # (S,)

    return U, V, c

In [ ]:
batch_size = 512 


def data_train_val(subset, ids, scanner):
    random.shuffle(ids)
    num_train = int(np.ceil(0.7 * len(ids))) 
    train_range, val_range = ids[:num_train], ids[num_train:]
    #print(f'for {scanner}, train range is {train_range}, val range is {val_range}')
    train_dataset = make_multi_WSI_dataset(subset, train_range, [scanner], train_or_test='Train', batch_size=batch_size)
    val_dataset = make_multi_WSI_dataset(subset, val_range, [scanner], train_or_test='Train', batch_size=batch_size)
    return train_dataset, val_dataset

akoya_data_train_subset1, akoya_data_val_subset1 = data_train_val('Subset1', [i for i in range(1,52+1)], 'Akoya')
akoya_data_train_subset3, akoya_data_val_subset3 = data_train_val('Subset3', [i for i in range(1,26+1)], 'Akoya')
leica_data_train, leica_data_val = data_train_val('Subset3', [i for i in range(1,26+1)], 'Leica')
leica_data_train, leica_data_val = data_train_val('Subset3', [i for i in range(1,26+1)], 'Leica')
philips_data_train, philips_data_val = data_train_val('Subset3', [i for i in range(1,26+1)], 'Philips')
olympus_data_train, olympus_data_val = data_train_val('Subset3', [i for i in range(1,26+1) if i != 20], 'Olympus')
zeiss_data_train, zeiss_data_val = data_train_val('Subset3', [1,5,6,7,8,9,10,11,12,13,14,16,21,23,25], 'Zeiss')

akoya_data_train = ConcatDataset([akoya_data_train_subset1, akoya_data_train_subset3])
akoya_data_val = ConcatDataset([akoya_data_val_subset1, akoya_data_val_subset3])

len_akoya_train = len(akoya_data_train)
len_akoya_val = len(akoya_data_val)

len_leica_train = len(leica_data_train)
len_leica_val = len(leica_data_val)

len_philips_train = len(philips_data_train) 
len_philips_val = len(philips_data_val)

len_olympus_train = len(olympus_data_train)
len_olympus_val = len(olympus_data_val)

len_zeiss_train = len(zeiss_data_train)
len_zeiss_val = len(zeiss_data_val)

len_train = len_akoya_train + len_leica_train + len_philips_train + len_olympus_train + len_zeiss_train
len_val = len_akoya_val + len_leica_val + len_philips_val + len_olympus_val + len_zeiss_val

#batch sizes for train
B_A_train = 128 #round(batch_size * len_akoya_train / len_train)
B_L_train = 64 #round(batch_size * len_leica_train / len_train)
B_P_train = 64 #round(batch_size * len_philips_train / len_train)
B_O_train = 64 #round(batch_size * len_olympus_train / len_train)
B_Z_train = 64 #batch_size - B_A_train - B_L_train - B_P_train - B_O_train


#batch sizes for validation
B_A_val = 128 #round(batch_size * len_akoya_val / len_val)
B_L_val = 64 #round(batch_size * len_leica_val / len_val)
B_P_val = 64 #round(batch_size * len_philips_val / len_val)
B_O_val = 64 #round(batch_size * len_olympus_val / len_val)
B_Z_val = 64 #batch_size - B_A_val - B_L_val - B_P_val - B_O_val


def make_loader(dataset, auto_batch_size):
    return DataLoader(dataset, batch_size=auto_batch_size, shuffle=True, num_workers=1, pin_memory=True, persistent_workers=True, prefetch_factor=4)

akoya_loader_train = make_loader(akoya_data_train, B_A_train)
leica_loader_train = make_loader(leica_data_train, B_L_train)
philips_loader_train = make_loader(philips_data_train, B_P_train)
olympus_loader_train = make_loader(olympus_data_train, B_O_train)
zeiss_loader_train = make_loader(zeiss_data_train, B_Z_train)



print("Train batches Akoya:", len(akoya_data_train), 'batch size:', B_A_train)
print("Train batches Leica:", len(leica_data_train), 'batch size:', B_L_train)
print("Train batches Philips:", len(philips_data_train), 'batch size:', B_P_train)
print("Train batches olympus:", len(olympus_data_train), 'batch size:', B_O_train)
print("Train batches Zeiss:", len(zeiss_data_train), 'batch size:', B_Z_train)
print('len train:', len_train)


akoya_loader_val = make_loader(akoya_data_val, B_A_val)
leica_loader_val = make_loader(leica_data_val, B_L_val)
philips_loader_val = make_loader(philips_data_val, B_P_val)
olympus_loader_val = make_loader(olympus_data_val, B_O_val)
zeiss_loader_val = make_loader(zeiss_data_val, B_Z_val)

print("Val batches Akoya:", len(akoya_data_val), 'batch size:', B_A_val)
print("Val batches Leica:", len(leica_data_val), 'batch size:', B_L_val)
print("Val batches Philips:", len(philips_data_val), 'batch size:', B_P_val)
print("Val batches olympus:", len(olympus_data_val), 'batch size:', B_O_val)
print("Val batches Zeiss:", len(zeiss_data_val), 'batch size:', B_Z_val)
print('len train:', len_val)

In [ ]:
for batch_akoya, batch_leica, batch_philips, batch_olympus, batch_zeiss in tqdm(zip(akoya_loader_train, leica_loader_train, philips_loader_train, olympus_loader_train, zeiss_loader_train),
                                                    desc=f"Epoch {epoch} - Training Multi Scanner",
                                                    total = min(len(akoya_loader_train), len(leica_loader_train), len(philips_loader_train), len(olympus_loader_train), len(zeiss_loader_train))):
            
    patch_akoya = batch_akoya['embedding'] 
    patch_leica = batch_leica['embedding']   
    patch_philips = batch_philips['embedding']
    patch_olympus = batch_olympus['embedding']
    patch_zeiss = batch_zeiss['embedding']
    
    t0 = time.time()
    
    t1 = time.time()
    sup_times.append(t1 - t0)
    
    
    # update tqdm description with average time and quality
    avg_sup = sum(sup_times) / len(sup_times)
    
    tq.set_description(f"OT losses | Avg time: {avg_sup:.3f}s")
        
        print('finish')

    